In [ ]:
import pandas as pd
import re
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# Encode target labels
df_clean = pd.read_csv('df_clean.csv')
df_clean['label'] = df_clean['class'].map({'non-suicide': 0, 'suicide': 1})

In [ ]:
# Remove any null or empty text rows
df_clean = df_clean[df_clean['text_clean_deep'].notnull()]
df_clean = df_clean[df_clean['text_clean_deep'].str.strip() != ""]

In [ ]:
# Define features and target
X = df_clean['text_clean_deep']
y = df_clean['label']

In [ ]:
# Train/test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print("X_train size:", X_train.shape)
print("X_test size:", X_test.shape)
print("y_train size:", y_train.shape)
print("y_test size:", y_test.shape)

In [ ]:
 # TF-IDF using unigrams
tfidf_uni = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    ngram_range=(1, 1)
)

X_train_tfidf_uni = tfidf_uni.fit_transform(X_train)
X_test_tfidf_uni = tfidf_uni.transform(X_test)

print("\nUnigram TF-IDF:")
print("Training matrix shape:", X_train_tfidf_uni.shape)
print("Testing matrix shape:", X_test_tfidf_uni.shape)

In [ ]:
# TF-IDF using unigrams and bigrams
tfidf_bi = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    ngram_range=(1, 2)
)

X_train_tfidf_bi = tfidf_bi.fit_transform(X_train)
X_test_tfidf_bi = tfidf_bi.transform(X_test)

In [ ]:
print("\nUnigram + Bigram TF-IDF:")
print("Training matrix shape:", X_train_tfidf_bi.shape)
print("Testing matrix shape:", X_test_tfidf_bi.shape)

In [ ]:
# Show some example features
uni_features = tfidf_uni.get_feature_names_out()
bi_features = tfidf_bi.get_feature_names_out()

In [ ]:
print("\nSample unigram features:")
print(uni_features[:20])

In [ ]:
print("\nSample unigram + bigram features:")
print(bi_features[:20])

##Train SVC model with mostly default hyperparameters

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score

In [ ]:
# Instantiate SVC model with mostly default hyperparameters
clf = SVC(probability=True, random_state=42)
clf.get_params()

In [ ]:
%%time

# Train model with tfidf features
clf.fit(X_train_tfidf_uni, y_train)

In [ ]:
# Tried Saving a Trained Model - models are saved using joblib instead of CSV since CSV cannot store trained models

# import joblib

# # Save model + vectorizer to avoid retraining (takes long)
# joblib.dump(clf, "svc_model.pkl")
# joblib.dump(tfidf_uni, "tfidf_uni_vectorizer.pkl")

# print("Saved successfully")

In [ ]:
%%time

# Display accuracy
predictions = clf.predict(X_test_tfidf_uni)
accuracy = round(accuracy_score(y_test, predictions)*100,3)
print(f"SVC Accuracy = {accuracy}%")

In [ ]:
# Display classification report
print("SVC Classification report")
print(classification_report(y_test,predictions))

In [ ]:
import matplotlib.pyplot as plt

# Compute confusion matrix
cm = confusion_matrix(y_test, predictions)

# Display confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()

print("SVC Confusion matrix")
plt.show()

### Try SVC again with Unigrams and Bigrams

In [ ]:
# Instantiate a second SVC model with mostly default hyperparameters
clf2 = SVC(probability=True, random_state=42)
clf2.get_params()

In [ ]:
%%time

# Train model with tfidf features
clf2.fit(X_train_tfidf_bi, y_train)

In [ ]:
%%time

# Display accuracy
predictions2 = clf2.predict(X_test_tfidf_bi)
accuracy2 = round(accuracy_score(y_test, predictions2)*100,3)
print(f"SVC Accuracy = {accuracy2}%")

In [ ]:
# Display classification report
print("SVC Classification report")
print(classification_report(y_test,predictions2))

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(y_test, predictions2)

# Display confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()

print("SVC Confusion matrix")
plt.show()

##Train Naive Bayes model

In [ ]:
from sklearn.naive_bayes import MultinomialNB

In [ ]:
# Instantiate Naive Bayes model
nb_clf = MultinomialNB()
nb_clf.get_params()

In [ ]:
%%time

# Train model with unigram TF-IDF features
nb_clf.fit(X_train_tfidf_uni, y_train)

In [ ]:
%%time

# Display accuracy
nb_predictions = nb_clf.predict(X_test_tfidf_uni)
nb_accuracy = round(accuracy_score(y_test, nb_predictions) * 100, 3)
print(f"Naive Bayes Accuracy = {nb_accuracy}%")

In [ ]:
# Display classification report
print("Naive Bayes Classification report")
print(classification_report(y_test, nb_predictions))

In [ ]:
# Display confusion matrix
nb_cm = confusion_matrix(y_test, nb_predictions)
nb_disp = ConfusionMatrixDisplay(confusion_matrix=nb_cm)
nb_disp.plot()
print("Naive Bayes Confusion matrix")
plt.show()

In [ ]:
# ROC-AUC
nb_proba = nb_clf.predict_proba(X_test_tfidf_uni)[:, 1]
nb_fpr, nb_tpr, _ = roc_curve(y_test, nb_proba)
nb_auc = roc_auc_score(y_test, nb_proba)

plt.figure(figsize=(7, 5))
plt.plot(nb_fpr, nb_tpr, label=f'Naive Bayes (AUC = {nb_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Naive Bayes ROC Curve')
plt.legend()
plt.show()

### Try Naive Bayes again with Unigrams and Bigrams

In [ ]:
# Instantiate a second Naive Bayes model
nb_clf2 = MultinomialNB()
nb_clf2.get_params()

In [ ]:
%%time

# Train model with unigram + bigram TF-IDF features
nb_clf2.fit(X_train_tfidf_bi, y_train)

In [ ]:
%%time

# Display accuracy
nb_predictions2 = nb_clf2.predict(X_test_tfidf_bi)
nb_accuracy2 = round(accuracy_score(y_test, nb_predictions2) * 100, 3)
print(f"Naive Bayes (Unigrams + Bigrams) Accuracy = {nb_accuracy2}%")

In [ ]:
# Display classification report
print("Naive Bayes (Unigrams + Bigrams) Classification report")
print(classification_report(y_test, nb_predictions2))

In [ ]:
# Display confusion matrix
nb_cm2 = confusion_matrix(y_test, nb_predictions2)
nb_disp2 = ConfusionMatrixDisplay(confusion_matrix=nb_cm2)
nb_disp2.plot()
print("Naive Bayes (Unigrams + Bigrams) Confusion matrix")
plt.show()

In [ ]:
# ROC-AUC
nb_proba2 = nb_clf2.predict_proba(X_test_tfidf_bi)[:, 1]
nb_fpr2, nb_tpr2, _ = roc_curve(y_test, nb_proba2)
nb_auc2 = roc_auc_score(y_test, nb_proba2)

plt.figure(figsize=(7, 5))
plt.plot(nb_fpr2, nb_tpr2, label=f'Naive Bayes Unigrams+Bigrams (AUC = {nb_auc2:.3f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Naive Bayes (Unigrams + Bigrams) ROC Curve')
plt.legend()
plt.show()

##Train Logistic Regression model

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
# Instantiate Logistic Regression model
lr_clf = LogisticRegression(max_iter=1000, random_state=42)
lr_clf.get_params()

In [ ]:
%%time

# Train model with unigram TF-IDF features
lr_clf.fit(X_train_tfidf_uni, y_train)

In [ ]:
# Display accuracy
lr_predictions = lr_clf.predict(X_test_tfidf_uni)
lr_accuracy = round(accuracy_score(y_test, lr_predictions) * 100, 3)
print(f"Logistic Regression Accuracy = {lr_accuracy}%")

In [ ]:
# Display classification report
print("Logistic Regression Classification report")
print(classification_report(y_test, lr_predictions))

In [ ]:
# Display confusion matrix
lr_cm = confusion_matrix(y_test, lr_predictions)
lr_disp = ConfusionMatrixDisplay(confusion_matrix=lr_cm)
lr_disp.plot()
print("Logistic Regression Confusion matrix")
plt.show()

In [ ]:
# ROC-AUC
lr_proba = lr_clf.predict_proba(X_test_tfidf_uni)[:, 1]
lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_proba)
lr_auc = roc_auc_score(y_test, lr_proba)

plt.figure(figsize=(7, 5))
plt.plot(lr_fpr, lr_tpr, label=f'Logistic Regression (AUC = {lr_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Logistic Regression ROC Curve')
plt.legend()
plt.show()

### Try Logistic Regression again with Unigrams and Bigrams

In [ ]:
# Instantiate a second Logistic Regression model
lr_clf2 = LogisticRegression(max_iter=1000, random_state=42)
lr_clf2.get_params()

In [ ]:
%%time

# Train model with unigram + bigram TF-IDF features
lr_clf2.fit(X_train_tfidf_bi, y_train)

In [ ]:
%%time

# Display accuracy
lr_predictions2 = lr_clf2.predict(X_test_tfidf_bi)
lr_accuracy2 = round(accuracy_score(y_test, lr_predictions2) * 100, 3)
print(f"Logistic Regression (Unigrams + Bigrams) Accuracy = {lr_accuracy2}%")

In [ ]:
# Display classification report
print("Logistic Regression (Unigrams + Bigrams) Classification report")
print(classification_report(y_test, lr_predictions2))

In [ ]:
# Display confusion matrix
lr_cm2 = confusion_matrix(y_test, lr_predictions2)
lr_disp2 = ConfusionMatrixDisplay(confusion_matrix=lr_cm2)
lr_disp2.plot()
print("Logistic Regression (Unigrams + Bigrams) Confusion matrix")
plt.show()

In [ ]:
# ROC-AUC
lr_proba2 = lr_clf2.predict_proba(X_test_tfidf_bi)[:, 1]
lr_fpr2, lr_tpr2, _ = roc_curve(y_test, lr_proba2)
lr_auc2 = roc_auc_score(y_test, lr_proba2)

plt.figure(figsize=(7, 5))
plt.plot(lr_fpr2, lr_tpr2, label=f'Logistic Regression Unigrams+Bigrams (AUC = {lr_auc2:.3f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Logistic Regression (Unigrams + Bigrams) ROC Curve')
plt.legend()
plt.show()

### Model Comparison

In [ ]:
# Compare all models side by side
svc_proba = clf.predict_proba(X_test_tfidf_uni)[:, 1]
svc_fpr, svc_tpr, _ = roc_curve(y_test, svc_proba)
svc_auc = roc_auc_score(y_test, svc_proba)

svc_proba2 = clf2.predict_proba(X_test_tfidf_bi)[:, 1]
svc_fpr2, svc_tpr2, _ = roc_curve(y_test, svc_proba2)
svc_auc2 = roc_auc_score(y_test, svc_proba2)

plt.figure(figsize=(10, 7))
plt.plot(svc_fpr,  svc_tpr,  label=f'SVC Unigrams (AUC = {svc_auc:.3f})')
plt.plot(svc_fpr2, svc_tpr2, label=f'SVC Unigrams+Bigrams (AUC = {svc_auc2:.3f})', linestyle='--')
plt.plot(nb_fpr,   nb_tpr,   label=f'Naive Bayes Unigrams (AUC = {nb_auc:.3f})')
plt.plot(nb_fpr2,  nb_tpr2,  label=f'Naive Bayes Unigrams+Bigrams (AUC = {nb_auc2:.3f})', linestyle='--')
plt.plot(lr_fpr,   lr_tpr,   label=f'Logistic Regression Unigrams (AUC = {lr_auc:.3f})')
plt.plot(lr_fpr2,  lr_tpr2,  label=f'Logistic Regression Unigrams+Bigrams (AUC = {lr_auc2:.3f})', linestyle='--')
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison — All Classical Models')
plt.legend(loc='lower right')
plt.show()

In [ ]:
# Summary table — all classical model variants
svc_accuracy  = round(accuracy_score(y_test, predictions)  * 100, 3)
svc_accuracy2 = round(accuracy_score(y_test, predictions2) * 100, 3)

summary = pd.DataFrame({
    'Model':        ['SVC (Unigrams)', 'SVC (Unigrams+Bigrams)',
                     'Naive Bayes (Unigrams)', 'Naive Bayes (Unigrams+Bigrams)',
                     'Logistic Regression (Unigrams)', 'Logistic Regression (Unigrams+Bigrams)'],
    'Accuracy (%)': [svc_accuracy,  svc_accuracy2,
                     nb_accuracy,   nb_accuracy2,
                     lr_accuracy,   lr_accuracy2],
    'AUC':          [round(svc_auc, 4), round(svc_auc2, 4),
                     round(nb_auc,  4), round(nb_auc2,  4),
                     round(lr_auc,  4), round(lr_auc2,  4)]
})
summary